# Modul KKA: Uninformed Search

## Tujuan praktikum

Pada praktikum ini, Anda akan merumuskan masalah pencarian, menelusuri urutan
kunjungan simpul, dan membandingkan hasil beberapa algoritma.

### Formulasi masalah

Contoh labirin menggunakan aturan berikut.

| Komponen | Definisi |
|---|---|
| Keadaan (*state*) | Koordinat `(baris, kolom)` pada sel yang dapat dilalui |
| Keadaan awal | `(0, 0)` |
| Aksi | Bergerak satu sel ke kanan, bawah, kiri, atau atas |
| Uji tujuan (*goal test*) | Koordinat sama dengan `(4, 4)` |
| Biaya lintasan $g(n)$ | Jumlah perpindahan dari keadaan awal; setiap perpindahan berbiaya 1 |

Gerakan tidak boleh keluar dari labirin atau menembus dinding. Satu keadaan dapat
dicapai melalui beberapa lintasan. Simpul pencarian menyimpan keadaan beserta
informasi lintasannya, seperti induk, kedalaman, dan biaya.

### Kriteria perbandingan

- **Kelengkapan (*complete*):** apakah algoritma menemukan solusi jika solusi ada?
- **Optimalitas:** apakah solusi yang ditemukan memiliki biaya minimum?
- **Waktu:** berapa banyak simpul yang diekspansi dan berapa lama pencarian berlangsung?
- **Memori:** berapa banyak informasi yang harus disimpan selama pencarian?

Dalam analisis kompleksitas, $b$ adalah jumlah cabang maksimum per simpul,
$d$ kedalaman solusi terdangkal, $m$ kedalaman maksimum pohon pencarian, dan
$\ell$ batas kedalaman pada DLS. Panjang lintasan dihitung dalam perpindahan:
`len(path) - 1`.

## Persiapan praktikum

Lengkapi identitas berikut. Petunjuk menjalankan Jupyter dan Google Colab tersedia di README.

| Nama | NRP |
|---|---|
| | |

### Mengenali bagian yang harus diisi

Setiap bagian tugas diberi komentar **`# TODO [kode]`**, diikuti penanda
**`________("kode")`** pada posisi jawaban. Nomor pada komentar dan penanda
sama, misalnya U01.

- Ganti **seluruh pemanggilan** `________("U01")`, termasuk tanda kurung dan
  kodenya, dengan ekspresi atau perintah Python yang diminta pada komentar.
- Untuk tugas satu fungsi penuh, tulis implementasi di antara komentar
  `# __ MULAI JAWABAN __` dan `# __ AKHIR JAWABAN __`.
- Kode di luar penanda sudah disediakan. Sel bertanda **PENGUJIAN** tidak perlu
  diubah; jalankan setelah fungsi latihan selesai diisi.

Contoh: jika diminta menghitung dua kali x, ubah `nilai = ________("X01")`
menjadi `nilai = 2 * x`.

Jalankan sel berurutan. Setelah mengubah fungsi, jalankan kembali sel definisi
dan sel pengujiannya. Pesan **BELUM DIISI** menyebut kode TODO yang masih kosong.
Jika `assert` gagal, periksa kembali jawaban Anda. Fungsi penanda di bawah
digunakan untuk menampilkan pesan tersebut dan tidak perlu diubah.

Seluruh impor ada pada sel persiapan. Setiap algoritma didefinisikan langsung pada bagian materinya, diikuti contoh pemanggilan dan latihan.

In [ ]:
from pathlib import Path
from collections import deque
from itertools import count
from time import perf_counter
import heapq
import random
import sys

import numpy as np

# Jalankan sel persiapan sebelum sel materi atau latihan.
module_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                    if (p / 'data/search_lab.py').is_file()), None)
if module_root is None:
    raise FileNotFoundError('Folder data/ tidak ditemukan. Ikuti petunjuk pada README.')
sys.path.insert(0, str(module_root.resolve()))

from data.search_lab import MAZE, START, GOAL, DIRECTIONS, animate_expansion
from data.utils import visualize_maze

random.seed(42)
np.random.seed(42)

In [ ]:
# FUNGSI PENANDA LATIHAN — tidak perlu diubah.
def ________(kode):
    raise NotImplementedError(f"TODO [{kode}] belum diisi. Lihat komentar TODO dengan kode yang sama.")

## 1. Depth-First Search (DFS)

DFS menelusuri satu cabang hingga tidak ada simpul baru yang dapat dikunjungi,
lalu kembali untuk mencoba cabang lain. Implementasi iteratif menggunakan
tumpukan (*stack*): simpul yang terakhir dimasukkan akan diproses lebih dulu.

### Cara kerja

1. Masukkan simpul awal ke tumpukan.
2. Ambil simpul teratas. Lewati simpul yang sudah dikunjungi.
3. Periksa apakah simpul tersebut merupakan tujuan.
4. Masukkan tetangganya yang belum dikunjungi, lalu ulangi.

Urutan memasukkan tetangga menentukan cabang yang ditelusuri lebih dulu. DFS
tidak menjamin lintasan terpendek. Pada graf berhingga, himpunan `visited`
mencegah kunjungan berulang dan memastikan pencarian berhenti. Pada ruang
pencarian berkedalaman tak hingga, DFS dapat terus menelusuri satu cabang.

### Contoh: penelusuran graf

Graf berikut memiliki akar A, dengan cabang ke B dan C. Fungsi `dfs` menampilkan
urutan kunjungan seluruh simpul yang dapat dicapai dari A.

In [ ]:
def dfs(adj: dict, start, visited=None, path=None):
    if visited is None:
        visited = set()
        path = []
        
    visited.add(start)
    path.append(start)
    
    for neighbor in adj.get(start, []):
        if neighbor not in visited:
            dfs(adj, neighbor, visited, path)
            
    return path

In [ ]:
adj = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': [],
    'F': []
}

dfs(adj, start='A')

### Latihan dasar: DFS pada labirin

Lengkapi `dfs_grid` untuk mencari lintasan dari `(0, 0)` ke `(4, 4)` pada labirin
`MAZE`. Nilai 0 menandai sel bebas dan nilai 1 menandai dinding. Gunakan tumpukan
dan himpunan `visited`. Hasil fungsi berupa daftar koordinat dari awal sampai tujuan.

Pertanyaan:
1. Mengapa DFS menghasilkan 12 langkah, sedangkan BFS menghasilkan 8 pada labirin ini?
2. Bagaimana perubahan urutan memasukkan tetangga memengaruhi lintasan DFS?
3. Apa yang dapat terjadi jika `visited` dihilangkan?

In [ ]:
# BAGIAN TUGAS — isi 2 penanda: U01, U02.
try:
    def dfs_grid(maze, start, goal):
        stack = [(start, [start])]
        visited = set()
        while stack:
            # TODO [U01]: Ambil satu elemen terakhir dari stack.
            current, path = ________("U01")
            if current in visited:
                continue
            if current == goal:
                return path
            # TODO [U02]: Tambahkan current ke himpunan visited.
            ________("U02")
            x, y = current
            for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]) and maze[nx][ny] == 0:
                    stack.append(((nx, ny), path + [(nx, ny)]))
        return None
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    maze = np.array(MAZE)
    start, goal = START, GOAL
    path_dfs = dfs_grid(maze, start, goal)
    assert path_dfs[0] == start and path_dfs[-1] == goal
    assert len(path_dfs) - 1 == 12
    print('LULUS DFS:', len(path_dfs) - 1, 'langkah')
    visualize_maze(maze, start, goal, path_dfs, algo='dfs')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 2. Breadth-First Search (BFS)

BFS menelusuri simpul per lapisan kedalaman: semua simpul pada kedalaman 1
diproses sebelum simpul pada kedalaman 2, dan seterusnya. Antrean (*queue*)
memastikan simpul yang lebih dulu dimasukkan diproses lebih dulu.

### Cara kerja

1. Masukkan simpul awal ke antrean dan tandai sebagai sudah dikunjungi.
2. Ambil simpul paling depan dan periksa tujuan.
3. Masukkan tetangga yang belum dikunjungi, lalu tandai agar tidak masuk dua kali.
4. Ulangi sampai tujuan ditemukan atau antrean kosong.

BFS menemukan solusi terdangkal jika jumlah cabang per simpul berhingga.
Solusi tersebut juga berbiaya minimum apabila biaya semua langkah sama.
Kebutuhan memorinya dapat besar karena banyak simpul pada satu lapisan harus disimpan.

### Contoh: penelusuran graf

In [ ]:
def bfs(adj, start):
    visited = set()
    res = []
    
    q = deque()
    visited.add(start)
    q.append(start)

    while q:
        curr = q.popleft()
        res.append(curr)

        for neighbor in adj[curr]:
            if neighbor not in visited:
                visited.add(neighbor)
                q.append(neighbor)
                
    return res

In [ ]:
adj = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': [],
    'F': []
}

bfs(adj, start='A')

### Latihan dasar: BFS pada labirin

Lengkapi `bfs_grid` pada labirin yang sama dengan latihan DFS. Titik awalnya
`(0, 0)` dan tujuannya `(4, 4)`. Gunakan antrean dan kembalikan daftar koordinat
lintasan. Bandingkan panjang hasilnya dengan hasil DFS.

Pertanyaan:
1. Mengapa urutan FIFO membuat BFS memproses simpul menurut kedalamannya?
2. Kapan lintasan BFS juga memiliki biaya minimum?
3. Apa akibatnya jika simpul baru ditandai saat dikeluarkan, bukan saat dimasukkan ke antrean?

In [ ]:
# BAGIAN TUGAS — isi 2 penanda: U03, U04.
try:
    def bfs_grid(maze, start, goal):
        q = deque([(start, [start])])
        visited = {start}
        while q:
            # TODO [U03]: Ambil satu elemen paling depan dari q.
            current, path = ________("U03")
            if current == goal:
                return path
            x, y = current
            for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                child = (nx, ny)
                if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]) and maze[nx][ny] == 0 and child not in visited:
                    # TODO [U04]: Tambahkan child ke visited sebelum masuk antrean.
                    ________("U04")
                    q.append((child, path + [child]))
        return None
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    maze = np.array(MAZE)
    start, goal = START, GOAL
    path_bfs = bfs_grid(maze, start, goal)
    assert len(path_bfs) - 1 == 8
    print('LULUS BFS:', len(path_bfs) - 1, 'langkah')
    visualize_maze(maze, start, goal, path_bfs, algo='bfs')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 3. Uniform-Cost Search (UCS)

UCS memilih simpul dengan biaya lintasan $g(n)$ terkecil. Algoritma ini cocok
untuk graf berbobot, ketika jumlah langkah tidak cukup untuk menentukan lintasan
termurah. Simpul yang dapat dicapai melalui lintasan lebih murah diperbarui biayanya.

### Cara kerja

1. Masukkan simpul awal ke antrean prioritas dengan biaya 0.
2. Keluarkan simpul dengan biaya terkecil.
3. Jika simpul tersebut adalah tujuan, kembalikan biaya dan lintasannya.
4. Untuk setiap tetangga, hitung biaya baru dan perbarui jika lebih murah.
5. Abaikan entri antrean yang biayanya sudah tidak sesuai dengan biaya terbaik.

UCS lengkap dan optimal jika jumlah cabang berhingga dan biaya setiap langkah
setidaknya $\epsilon>0$. Tanpa heuristik, UCS juga dapat menelusuri banyak
lintasan murah yang tidak mengarah ke tujuan.

### Contoh: graf berbobot

In [ ]:
def reconstruct_path(visited, start, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = visited[current][1]
    return path[::-1]


def uniform_cost_search(graph, start, goal):
    priority_queue = [(0, start)]
    visited = {start: (0, None)}
    
    while priority_queue:
        current_cost, current_node = heapq.heappop(priority_queue)
        
        if current_node == goal:
            return current_cost, reconstruct_path(visited, start, goal)
            
        for neighbor, cost in graph[current_node]:
            total_cost = current_cost + cost
            
            if neighbor not in visited or total_cost < visited[neighbor][0]:
                visited[neighbor] = (total_cost, current_node)
                heapq.heappush(priority_queue, (total_cost, neighbor))
                
    return None

In [ ]:
adj_ucs = {
    'A': [('B', 1), ('C', 4)],
    'B': [('C', 2), ('D', 5)],
    'C': [('D', 1)],
    'D': []
}

print(uniform_cost_search(adj_ucs, 'A', 'D'))

### Latihan dasar: UCS pada grid berbobot

Cari lintasan berbiaya minimum dari `(0, 0)` ke `(4, 4)`. Nilai positif pada sel
adalah biaya memasuki sel tersebut; nilai 0 berarti rintangan. Biaya awal adalah 0,
sehingga nilai sel awal tidak ikut dijumlahkan. Lengkapi `ucs_grid` agar
mengembalikan total biaya dan daftar koordinat lintasan.

Pertanyaan:
1. Bagaimana perhitungan berubah jika biaya sel awal ikut dihitung?
2. Mengapa tujuan diperiksa ketika simpul dikeluarkan dari antrean prioritas?
3. Mengapa lintasan dengan langkah paling sedikit belum tentu paling murah?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: U05.
try:
    def ucs_grid(grid, start, goal):
        rows = len(grid)
        cols = len(grid[0])
        
        pq = [(0, start, [start])]
        visited_costs = {start: 0}
        
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        while pq:
            cost, curr, path = heapq.heappop(pq)

            if curr == goal:
                return cost, path

            if cost > visited_costs.get(curr, float('inf')):
                continue

            r, c = curr
            for dr, dc in directions:
                nr, nc = r + dr, c + dc
                
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != 0:
                    # TODO [U05]: Jumlahkan biaya saat ini dan biaya memasuki sel berikutnya.
                    new_cost = ________("U05")
                    
                    if new_cost < visited_costs.get((nr, nc), float('inf')):
                        visited_costs[(nr, nc)] = new_cost
                        heapq.heappush(pq, (new_cost, (nr, nc), path + [(nr, nc)]))
                        
        return float('inf'), []
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    grid = [
        [1, 3, 1, 1, 1],
        [1, 0, 5, 0, 1],
        [2, 2, 1, 1, 8],
        [1, 0, 0, 4, 1],
        [1, 1, 1, 2, 1]
    ]

    start = (0, 0)
    goal = (4, 4)

    min_cost, best_path = ucs_grid(grid, start, goal)

    print(f"Biaya Minimum: {min_cost}")
    print(f"Jalur: {best_path}")
    assert min_cost == 10
    assert min_cost == sum(grid[r][c] for r,c in best_path[1:])
    print('LULUS UCS berbobot')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 4. Depth-Limited Search (DLS)

DLS menjalankan DFS dengan batas kedalaman $\ell$. Akar berada pada kedalaman 0.
Simpul pada batas tetap diperiksa sebagai tujuan, tetapi tidak diekspansi lagi.

Batas kedalaman mencegah pencarian terus masuk ke cabang yang terlalu dalam.
Namun, solusi di luar batas tidak akan ditemukan. DLS juga tidak menjamin
solusi optimal. Kebutuhan memori penelusuran umumnya $O(b\ell)$.

Pada graf yang memiliki beberapa lintasan menuju keadaan yang sama, pemeriksaan
siklus dapat dilakukan pada lintasan saat ini. `visited` global tanpa informasi
kedalaman dapat membuang lintasan lain yang mencapai keadaan itu lebih dangkal.

### Contoh: pencarian dengan batas kedalaman

Gunakan graf berikut untuk mencari F dari A dengan batas kedalaman 2.
Perhatikan cabang melalui D: pencarian tidak perlu menelusurinya sampai L.

In [ ]:
# Representasi Graf
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H', 'I'],
    'E': [],
    'F': [],
    'G': [],
    'H': ['J'],
    'I': [],
    'J': ['K'],
    'K': ['L'],
    'L': []
}

In [ ]:
def dls(graph, start, goal, limit):
    stack = [(start, [start], 0)]  # (node, path, depth)

    while stack:
        node, path, depth = stack.pop()

        if node == goal:
            return path

        if depth < limit:
            for neighbor in reversed(graph[node]):
                stack.append((neighbor, path + [neighbor], depth + 1))

    return None

In [ ]:
dls(graph, 'A', 'F', limit=2)

### Latihan dasar: DLS pada labirin

Lengkapi `dls_grid` pada labirin `MAZE`, dari `START` ke `GOAL`. Cegah siklus
dengan memeriksa koordinat pada lintasan saat ini. Jalankan fungsi dengan batas
kedalaman 7 dan 12.

Pertanyaan:
1. Mengapa tujuan tidak ditemukan dengan batas 7?
2. Mengapa `visited` global dapat menghilangkan lintasan yang masih layak dicoba?
3. Apakah memperbesar batas membuat DLS selalu menghasilkan lintasan terpendek?

In [ ]:
# BAGIAN TUGAS — isi 2 penanda: U06, U07.
try:
    def dls_grid(maze, start, goal, limit):
        stack = [(start, 0, [start])]
        while stack:
            node, depth, path = stack.pop()
            if node == goal:
                return path
            # TODO [U06]: Periksa apakah depth sudah mencapai limit.
            if ________("U06"):
                continue
            x, y = node
            for dx, dy in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                child = (x + dx, y + dy)
                nx, ny = child
                # TODO [U07]: Pastikan child belum terdapat pada path.
                if 0 <= nx < len(maze) and 0 <= ny < len(maze[0]) and maze[nx][ny] == 0 and ________("U07"):
                    stack.append((child, depth + 1, path + [child]))
        return None

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    assert dls_grid(MAZE, START, GOAL, 7) is None
    assert dls_grid(MAZE, START, GOAL, 12) is not None
    print('LULUS DLS:', dls_grid(MAZE, START, GOAL, 12))
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 5. Iterative Deepening Search (IDS)

IDS menjalankan DLS berulang kali dengan batas kedalaman 0, 1, 2, dan seterusnya.
Pencarian berhenti saat solusi ditemukan atau batas maksimum tercapai.

Dengan jumlah cabang berhingga, IDS menemukan solusi terdangkal. Jika biaya
langkah seragam, solusi itu juga optimal. Kebutuhan memori penelusurannya
seperti DFS, tetapi simpul dekat akar diekspansi ulang pada setiap pengulangan.

Fungsi `ids` menggunakan `dls` yang sudah didefinisikan pada bagian sebelumnya. Jalankan bagian DLS sebelum mencoba IDS.

In [ ]:
# Implementasi IDS
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H', 'I'],
    'E': [],
    'F': [],
    'G': [],
    'H': ['J'],
    'I': [],
    'J': ['K'],
    'K': ['L'],
    'L': []
}

In [ ]:
def ids(graph, start, goal, max_depth):
    for limit in range(max_depth + 1):
        result = dls(graph, start, goal, limit)
        if result:
            print(f"Goal ditemukan di depth {limit}")
            return result
    print("Goal tidak ditemukan")
    return None

In [ ]:
ids(graph, 'A', 'F', 5)

### Latihan dasar: IDS pada labirin

Lengkapi `ids_grid` agar memanggil DLS dari batas 0 sampai `max_depth`, termasuk
batas akhirnya. Gunakan labirin `MAZE`, dari `START` ke `GOAL`, dan
`max_depth=12`. Kembalikan lintasan pertama yang ditemukan.

Pertanyaan:
1. Mengapa pencarian dimulai dengan batas 0?
2. Mengapa jumlah ekspansi IDS mencakup kunjungan berulang?
3. Mengapa solusi pertama IDS optimal jika biaya setiap langkah sama?

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: U08.
try:
    def ids_grid(maze, start, goal, max_depth):
        # TODO [U08]: Buat urutan batas dari 0 sampai max_depth, termasuk batas akhirnya.
        for limit in ________("U08"):
            found = dls_grid(maze, start, goal, limit)
            if found is not None:
                return found
        return None

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    path_ids = ids_grid(MAZE, START, GOAL, max_depth=12)
    assert len(path_ids) - 1 == 8
    print('LULUS IDS:', path_ids)
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 6. Bidirectional Search

![](data/bidirectional.png)

Pencarian dua arah menjalankan pencarian dari keadaan awal dan dari tujuan.
Ketika kedua sisi bertemu, lintasannya digabungkan. Pada graf tak berarah dengan
biaya langkah seragam, kedua sisi dapat menggunakan BFS.

Implementasi berikut memproses satu lapisan penuh sebelum memeriksa pertemuan.
Jika ada beberapa titik temu pada lapisan itu, pilih yang jumlah jaraknya paling
kecil. Simpan induk pada kedua sisi untuk menyusun kembali lintasan.

### Syarat penggunaan

Bidirectional BFS lengkap dan optimal pada graf tak berarah dengan biaya langkah
seragam, jika ekspansi dan pemilihan titik temu mengikuti aturan BFS. Pada graf
berarah, pencarian dari tujuan harus mengikuti sisi terbalik (*predecessor*).
Graf berbobot memerlukan penanganan biaya yang berbeda.

Pada pohon pencarian yang cukup seimbang, waktu dan memorinya sekitar
$O(b^{d/2})$. Kedua sisi tetap perlu menyimpan antrean dan informasi kunjungan.

![](data/bidirectional_implementasi.png)

Cari lintasan dari simpul 1 ke simpul 14 pada graf berikut.

In [ ]:
graph_bidirectional = {
    0: [4],
    1: [4],
    2: [5],
    3: [5],
    4: [0,1,6],
    5: [2,3,6],
    6: [4,5,7],
    7: [6,8],
    8: [7,9,10],
    9: [8,11,12],
    10: [8,13,14],
    11: [9],
    12: [9],
    13: [10],
    14: [10]
}

In [ ]:
def bidirectional_path(neighbors, start, goal, stats=None):
    """BFS dua arah per LAPISAN; graf tak berarah, biaya tiap edge = 1."""
    if stats is not None:
        stats.update(expanded=0, frontier_peak=2, trace=[])
    if start == goal:
        return [start]
    fronts = [deque([start]), deque([goal])]
    parents = [{start: None}, {goal: None}]
    distances = [{start: 0}, {goal: 0}]
    while all(fronts):
        for side in (0, 1):
            meetings = []
            for _ in range(len(fronts[side])):
                node = fronts[side].popleft()
                if stats is not None:
                    stats['expanded'] += 1
                    stats['trace'].append((side, node))
                for child in neighbors(node):
                    if child not in parents[side]:
                        parents[side][child] = node
                        distances[side][child] = distances[side][node] + 1
                        fronts[side].append(child)
                        if child in parents[1 - side]:
                            meetings.append(child)
                if stats is not None:
                    stats['frontier_peak'] = max(stats['frontier_peak'], sum(map(len, fronts)))
            if meetings:
                meeting = min(meetings, key=lambda n: distances[0][n] + distances[1][n])
                left, right, node = [], [], meeting
                while node is not None:
                    left.append(node)
                    node = parents[0][node]
                node = parents[1][meeting]
                while node is not None:
                    right.append(node)
                    node = parents[1][node]
                return left[::-1] + right
    return None


def bidirectional_bfs(graph_bidirectional, start, goal):
    return bidirectional_path(lambda node: graph_bidirectional.get(node, []), start, goal)

In [ ]:
bidirectional_bfs(graph_bidirectional, 1, 14)

### Latihan dasar: langkah kuda pada papan catur

Cari lintasan kuda dari `(0, 0)` ke `(7, 7)` pada papan 8 × 8 menggunakan
pencarian dua arah. Koordinat dimulai dari 0. Gerakan kuda mengubah satu koordinat
sebesar 2 dan koordinat lain sebesar 1; tujuan gerakan harus tetap berada di papan.

Lengkapi pemanggilan pencarian dan fungsi penggabung lintasan.

Pertanyaan:
1. Mengapa pencarian memproses satu lapisan penuh sebelum memilih titik temu?
2. Bagaimana cara menggabungkan lintasan tanpa menulis titik temu dua kali?
3. Apa yang perlu diubah jika gerakan hanya diizinkan dalam satu arah?

In [ ]:
start = (0, 0)
goal = (7, 7)

In [ ]:
def get_knight_moves(pos):
    x, y = pos
    
    moves = [
        (x+2, y+1), (x+2, y-1),
        (x-2, y+1), (x-2, y-1),
        (x+1, y+2), (x+1, y-2),
        (x-1, y+2), (x-1, y-2)
    ]
    
    valid_moves = []
    for nx, ny in moves:
        if 0 <= nx < 8 and 0 <= ny < 8:
            valid_moves.append((nx, ny))
    
    return valid_moves

In [ ]:
# BAGIAN TUGAS — isi 3 penanda: U09, U10, U11.
try:
    def bidirectional_knight(start, goal):
        return bidirectional_path(
            # TODO [U09]: Fungsi pembangkit langkah kuda.
            ________("U09"),
            # TODO [U10]: Koordinat awal pencarian.
            ________("U10"),
            # TODO [U11]: Koordinat tujuan pencarian.
            ________("U11"),
        )
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# BAGIAN TUGAS — isi 1 penanda: U12.
try:
    def build_path(meeting, visited_start, visited_goal):
        left, right = [], []
        node = meeting
        while node is not None:
            left.append(node)
            node = visited_start[node]
        # TODO [U12]: Mulai dari parent meeting pada sisi goal.
        node = ________("U12")
        while node is not None:
            right.append(node)
            node = visited_goal[node]
        return left[::-1] + right

    # PENGUJIAN — jalankan setelah fungsi di atas diisi; tidak perlu diubah.
    assert build_path('B', {'A': None, 'B': 'A'}, {'C': None, 'B': 'C'}) == ['A', 'B', 'C']
    print('LULUS penyusunan lintasan')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

In [ ]:
# PENGUJIAN — jalankan setelah fungsi latihan diisi; tidak perlu diubah.
try:
    path = bidirectional_knight(start, goal)

    print("Lintasan:", path)
    print("Jumlah langkah:", len(path)-1 if path else None)
    assert len(path) - 1 == 6
    assert all(b in get_knight_moves(a) for a, b in zip(path, path[1:]))
    print('LULUS bidirectional kuda')
except NotImplementedError as error:
    print("BELUM DIISI:", error)

## 7. Perbandingan pada labirin yang sama

Jalankan semua metode dengan `MAZE`, `START`, dan `GOAL` yang sama. Setiap
perpindahan berbiaya 1. Urutan tetangga adalah kanan, bawah, kiri, atas; pada
DFS, tetangga yang terakhir masuk tumpukan diproses lebih dulu.

- `cost`: biaya lintasan, sekaligus jumlah langkah pada labirin ini.
- `expanded`: jumlah simpul yang tetangganya dibangkitkan, tidak termasuk tujuan.
  Pada IDS, ekspansi ulang ikut dihitung.
- `frontier_peak`: jumlah entri antrean atau tumpukan terbanyak. Untuk DLS/IDS,
  kolom ini menunjukkan panjang tumpukan rekursi maksimum.

Ukuran tersebut tidak mencakup seluruh memori program, misalnya penyimpanan
lintasan dan `visited`. Waktu eksekusi dapat berbeda antarkomputer.
Pada animasi, biru menunjukkan simpul yang diekspansi dan hijau menunjukkan lintasan hasil.

Sel berikut mendefinisikan fungsi pembanding beserta penghitungnya. Definisi algoritma dan cara menghitung ekspansi dapat dibaca langsung sebelum tabel hasil.

In [ ]:
def search_result(path, cost, expanded, frontier_peak, trace):
    return dict(path=path, cost=cost, expanded=expanded,
                frontier_peak=frontier_peak, trace=trace)


def grid_neighbors(grid, node):
    r, c = node
    for dr, dc in DIRECTIONS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
            yield (nr, nc), 1


def ucs_counted(start, goal, neighbors):
    """UCS pembanding; ekspansi tidak menghitung tujuan atau entri usang."""
    serial = count()
    frontier = [(0, next(serial), start)]
    costs, parents = {start: 0}, {start: None}
    trace, peak = [], 1
    while frontier:
        g, _, node = heapq.heappop(frontier)
        if g != costs[node]:
            continue
        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parents[node]
            return search_result(path[::-1], g, len(trace), peak, trace)
        trace.append(node)
        for child, step_cost in neighbors(node):
            if step_cost <= 0:
                raise ValueError('Biaya langkah harus positif')
            new_g = g + step_cost
            if new_g < costs.get(child, float('inf')):
                costs[child], parents[child] = new_g, node
                heapq.heappush(frontier, (new_g, next(serial), child))
        peak = max(peak, len(frontier))
    return search_result(None, float('inf'), len(trace), peak, trace)


def maze_search(method, grid=MAZE, start=START, goal=GOAL, limit=12):
    neighbors = lambda node: grid_neighbors(grid, node)
    if method == 'UCS':
        return ucs_counted(start, goal, neighbors)
    if method in ('DLS', 'IDS'):
        trace, peak = ([], 1)

        def visit(node, path, remaining):
            nonlocal peak
            peak = max(peak, len(path))
            if node == goal:
                return path
            if remaining == 0:
                return None
            trace.append(node)
            for child, _ in neighbors(node):
                if child not in path:
                    found = visit(child, path + [child], remaining - 1)
                    if found is not None:
                        return found
            return None
        for bound in range(limit + 1) if method == 'IDS' else (limit,):
            path = visit(start, [start], bound)
            if path is not None:
                return search_result(path, len(path) - 1, len(trace), peak, trace)
        return search_result(None, float('inf'), len(trace), peak, trace)
    if method not in ('DFS', 'BFS'):
        raise ValueError(method)
    frontier = deque([(start, [start])])
    visited, trace, peak = (set(), [], 1)
    while frontier:
        node, path = frontier.pop() if method == 'DFS' else frontier.popleft()
        if node in visited:
            continue
        visited.add(node)
        if node == goal:
            return search_result(path, len(path) - 1, len(trace), peak, trace)
        trace.append(node)
        for child, _ in neighbors(node):
            if child not in visited:
                frontier.append((child, path + [child]))
        peak = max(peak, len(frontier))
    return search_result(None, float('inf'), len(trace), peak, trace)

In [ ]:
maze_results = {}
print('Metode | Langkah | Ekspansi | Maks. frontier/tumpukan | Detik')
for method in ('DFS', 'BFS', 'UCS', 'DLS', 'IDS'):
    before = perf_counter()
    row = maze_search(method)
    maze_results[method] = row
    print(method, row['cost'], row['expanded'], row['frontier_peak'], f"{perf_counter()-before:.6f}", sep=' | ')
assert maze_results['BFS']['cost'] == 8
print('LULUS perbandingan maze')
display(animate_expansion(MAZE, maze_results['DFS']))

In [ ]:
bidi_stats = {}
bidi_maze = bidirectional_path(lambda node: [child for child, _ in grid_neighbors(MAZE, node)], START, GOAL, bidi_stats)
assert len(bidi_maze)-1 == 8
print('Pencarian dua arah pada labirin yang sama:', len(bidi_maze)-1, 'langkah;', bidi_stats['expanded'], 'ekspansi dua arah')
print('Penghitung mencakup akar kedua arah; maksimum frontier:', bidi_stats['frontier_peak'])

### Latihan lanjutan: pengaruh urutan dan biaya

1. Balik urutan tetangga pada DFS. Catat perubahan lintasan dan jumlah ekspansi.
2. Ubah satu koridor menjadi lebih mahal, lalu bandingkan BFS dan UCS.
   Pisahkan jumlah langkah dari total biaya dalam laporan Anda.
3. Jelaskan kapan Anda akan memilih BFS, UCS, atau IDS berdasarkan hasil percobaan.

## 8. Ringkasan

Tabel ini menggunakan analisis pohon pencarian. Simbol $C^*$ menyatakan biaya
solusi optimal dan $\epsilon>0$ adalah batas bawah biaya langkah.

| Algoritma | Lengkap | Optimal | Waktu | Memori penelusuran |
|---|---|---|---|---|
| DFS | Pada graf berhingga dengan `visited` | Tidak | $O(b^m)$ | $O(bm)$ |
| BFS | Jika $b$ berhingga | Jika biaya seragam | $O(b^{d+1})$ | $O(b^{d+1})$ |
| UCS | Jika $b$ berhingga dan biaya $\ge\epsilon$ | Ya | $O(b^{1+\lfloor C^*/\epsilon\rfloor})$ | Sama dengan batas waktu |
| DLS | Jika $\ell\ge d$ dan $b$ berhingga | Tidak secara umum | $O(b^\ell)$ | $O(b\ell)$ |
| IDS | Jika $b$ berhingga | Jika biaya seragam | $O(b^d)$ | $O(bd)$ |
| Bidirectional BFS | Jika pencarian balik dapat dilakukan | Jika biaya seragam dan titik temu dipilih dengan benar | $O(b^{d/2})$ pada pencarian seimbang | $O(b^{d/2})$ pada pencarian seimbang |

Batas BFS di atas memperhitungkan pembangkitan lapisan setelah kedalaman solusi.
Pada graf eksplisit dengan daftar ketetanggaan dan `visited`, DFS/BFS membutuhkan
waktu $O(V+E)$ dan penyimpanan $O(V)$. Implementasi yang menyimpan salinan lintasan
pada setiap entri membutuhkan memori tambahan.

## Referensi

- Russell, S. & Norvig, P. *Artificial Intelligence: A Modern Approach*, edisi ke-4, Bab 3.
  [Daftar isi](https://aima.cs.berkeley.edu/contents.html).
- [Kode pendamping AIMA](https://github.com/aimacode/aima-python).